# 07 — Hybrid Search: Reciprocal Rank Fusion (RRF)
**Project:** Semantic Book Recommender — IT4142 HUST  
**Input:**
- `data/processed/books_with_emotions.csv`
- `models/bm25_index.pkl`
- `data/chroma_db/` (BGE-small embeddings)

**Output:**
- Hybrid search function combining Sparse (BM25) and Dense (BGE-small) retrieval using RRF

---
## Reciprocal Rank Fusion (RRF)
RRF is an algorithmic fusion technique that combines multiple ranked lists into a single ranked list based on the reciprocal of the rank of each document:
$$\text{RRF}(d) = \sum_{r \in R} \frac{1}{k + \text{rank}_r(d)}$$
Where $k=60$ is a constant and $R$ is the set of rankers.

## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import pickle, json, time
import scipy.sparse as sp
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

DATA_PATH   = Path('data/processed/books_with_emotions.csv')
MODEL_PATH  = Path('models')
CHROMA_PATH = Path('data/chroma_db')
EVAL_PATH   = Path('data/eval/test_queries.json')
REPORT_PATH = Path('reports')
FIGURE_PATH = Path('reports/figures')
FIGURE_PATH.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 150,
    'figure.facecolor': 'white', 'axes.facecolor': '#F9F9F9',
    'axes.spines.top': False, 'axes.spines.right': False,
})
MODEL_COLORS = {
    'TF-IDF'   : '#6B8CBA',
    'BM25'     : '#F4A261',
    'Semantic' : '#2A9D8F',
    'Hybrid'   : '#E76F51',
    'Reranking': '#8338EC',
}
print('Setup OK')

Setup OK


## 1. Load Data & Models

In [2]:
df = pd.read_csv(DATA_PATH)
df['isbn13'] = df['isbn13'].astype(str)
print(f'Books: {len(df):,}')

Books: 11,606


In [3]:
# --- BM25 ---
with open(MODEL_PATH / 'bm25_index.pkl', 'rb') as f:
    bm25 = pickle.load(f)
print(f'BM25 loaded  | corpus: {bm25.corpus_size:,}')

# --- BGE-small + ChromaDB ---
embed_model = SentenceTransformer('BAAI/bge-small-en-v1.5')
chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_PATH),
    settings=Settings(anonymized_telemetry=False),
)
collection = chroma_client.get_collection('books')
print(f'ChromaDB loaded | count: {collection.count():,}')

BGE_PREFIX = 'Represent this sentence for searching relevant passages: '

BM25 loaded  | corpus: 11,606


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ChromaDB loaded | count: 11,606


## 2. Base Search Functions

In [4]:
def search_bm25_ids(query: str, top_k: int) -> list[str]:
    """Return ordered list of isbn13 strings."""
    scores = bm25.get_scores(query.lower().split())
    idx = np.argsort(scores)[::-1][:top_k]
    return df.iloc[idx]['isbn13'].tolist()


def search_semantic_ids(query: str, top_k: int) -> list[str]:
    """Return ordered list of isbn13 strings."""
    q_emb = embed_model.encode(
        [BGE_PREFIX + query], normalize_embeddings=True
    )
    results = collection.query(
        query_embeddings=q_emb.tolist(),
        n_results=top_k,
        include=['distances'],
    )
    return results['ids'][0]


def get_book_metadata(isbn_list: list[str]) -> pd.DataFrame:
    """Fetch full book rows for a list of isbn13 values, preserving order."""
    isbn_set = set(isbn_list)
    subset = df[df['isbn13'].isin(isbn_set)].copy()
    # Preserve rank order
    order = {isbn: i for i, isbn in enumerate(isbn_list)}
    subset['_order'] = subset['isbn13'].map(order)
    return subset.sort_values('_order').drop(columns='_order').reset_index(drop=True)

print('Base search functions ready')

Base search functions ready


## 3. Reciprocal Rank Fusion (RRF) Core

In [5]:
def reciprocal_rank_fusion(
    ranked_lists: list[list[str]],
    k: int = 60,
    weights: list[float] = None,
) -> list[tuple[str, float]]:
    """
    Merge multiple ranked lists using Reciprocal Rank Fusion.

    Args:
        ranked_lists : list of ranked doc-id lists (highest rank = index 0)
        k            : RRF constant. Higher k → less penalty for lower ranks.
                       Standard default = 60 (Cormack et al. 2009)
        weights      : optional per-list weights (default: equal weights)

    Returns:
        List of (doc_id, rrf_score) sorted by descending score
    """
    if weights is None:
        weights = [1.0] * len(ranked_lists)
    assert len(weights) == len(ranked_lists)

    scores: dict[str, float] = defaultdict(float)
    for ranked_list, w in zip(ranked_lists, weights):
        for rank, doc_id in enumerate(ranked_list):
            scores[doc_id] += w * (1.0 / (k + rank + 1))

    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def search_hybrid_rrf(
    query: str,
    top_k: int = 10,
    candidate_pool: int = 50,
    k: int = 60,
    weights: list[float] = None,   # [bm25_weight, dense_weight]
) -> pd.DataFrame:
    """
    Hybrid retrieval: BM25 + BGE-small fused with RRF.

    Steps:
      1. BM25 → top-N candidates
      2. BGE-small → top-N candidates
      3. RRF merge → top-K final results
    """
    bm25_ids   = search_bm25_ids(query, top_k=candidate_pool)
    dense_ids  = search_semantic_ids(query, top_k=candidate_pool)

    fused = reciprocal_rank_fusion(
        [bm25_ids, dense_ids],
        k=k,
        weights=weights,
    )

    top_ids = [doc_id for doc_id, _ in fused[:top_k]]
    top_scores = {doc_id: score for doc_id, score in fused[:top_k]}

    result = get_book_metadata(top_ids).copy()
    result['rrf_score'] = result['isbn13'].map(top_scores).round(6)
    return result


# Smoke test
test = search_hybrid_rrf(
    'a heartbreaking story about family secrets in rural America', top_k=5
)
print('Hybrid RRF smoke test:')
test[['title', 'categories', 'rrf_score']]

Hybrid RRF smoke test:


,title,categories,rrf_score
0,Every Cloak Rolled in Blood,Fiction,0.027237
1,Taboo (16pt Large Print Edition),Unknown,0.026686
2,Daydreams and Nightmares,History,0.025564
3,Watching Glass Shatter,Identity (Psychology),0.025253
4,Always Time to Die,Governors,0.023720


## 9. Qualitative Demo — Vocabulary Mismatch Queries

Đây là bộ queries để **demo trực tiếp** trong báo cáo/slide:  
query dùng ngôn ngữ khác hoàn toàn so với mô tả sách.

In [6]:
# Load TF-IDF để so sánh
with open(MODEL_PATH / 'tfidf_vectorizer.pkl', 'rb') as f:
    vectorizer = pickle.load(f)
tfidf_matrix = sp.load_npz(MODEL_PATH / 'tfidf_matrix.npz')

def search_tfidf(query, top_k=5):
    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, tfidf_matrix).flatten()
    idx = np.argsort(scores)[::-1][:top_k]
    res = df.iloc[idx][['title', 'categories']].copy()
    res['score'] = scores[idx].round(4)
    return res.reset_index(drop=True)

MISMATCH_QUERIES = [
    'someone losing their mind slowly',           # → "psychological deterioration"
    'a book about a troubled youth finding hope', # → "coming of age", "redemption"
    'far east murder mystery',                    # → "Japan detective", "Tokyo crime"
]

for q in MISMATCH_QUERIES:
    print(f'\n{"="*60}')
    print(f'Query: "{q}"')
    print('\n[TF-IDF]')
    tf = search_tfidf(q, 3)
    print(tf.to_string(index=False))
    print('\n[Hybrid RRF]')
    hy = search_hybrid_rrf(q, top_k=3)[['title', 'categories', 'rrf_score']]
    print(hy.to_string(index=False))


Query: "someone losing their mind slowly"

[TF-IDF]
                     title       categories  score
The Secret in the Matchbox          Dragons 0.2304
      The Borrowers Afield Juvenile Fiction 0.2043
                   Thinner          Fiction 0.1692

[Hybrid RRF]
                                 title categories  rrf_score
 What to Do When the Mind Troubles You  Self-Help   0.027013
                 The Imaginary Husband    Fiction   0.024017
Mindfulness Bedtime Stories for Adults    Unknown   0.023932

Query: "a book about a troubled youth finding hope"

[TF-IDF]
          title           categories  score
Call It Courage     Juvenile Fiction 0.1835
       Chaotics Business & Economics 0.1657
  Youth Studies       Social Science 0.1570

[Hybrid RRF]
               title                categories  rrf_score
Finding My Lost Life Biography & Autobiography   0.031319
                Hope Biography & Autobiography   0.028405
         Brave Steps                 Self-Help   0.027799
